# Avaliação do modelo e do sistema

A avaliação separa três objetos: perda da LLM, qualidade das respostas brutas e funcionamento das regras de orquestração. Testes de regras não medem aprendizado. Documentos recuperados não equivalem a citações corretas.

As cinco respostas de teste são comparadas entre modelo base e adaptador selecionado pela validação. Não existe avaliação clínica especializada neste estudo.

In [1]:
from pathlib import Path
import json, os, sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*", module="tqdm.auto")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["HF_HOME"] = str(ROOT / ".hf-cache")
result = json.loads(Path("docs/results/training.json").read_text(encoding="utf-8"))
print("Perda base:", result["base_test_loss"])
print("Perda adaptada:", result["adapted_test_loss"])
print("Recall lexical base:", result["base_lexical_recall"])
print("Recall lexical adaptado:", result["adapted_lexical_recall"])
for row in result["comparisons"]:
    print("\n", row["id"], "\nReferência:", row["reference"], "\nBase:", row["base"], "\nAdaptado:", row["adapted"])

Perda base: 3.3887380599975585
Perda adaptada: 3.1454211235046388
Recall lexical base: 0.013333333333333332
Recall lexical adaptado: 0.023859649122807018

 FAQ-006 
Referência: Associar cada afirmação institucional ao código, título e versão do protocolo recuperado. Separar fatos do prontuário, trechos de protocolo e limitações da interpretação. 
Base: Respond in portuguese usando apenas as evidences. No confirme diagnostic n prescreva. Se faltar evidência declare insuficiência. Question: How to Cite Asesics Usadas na resposta? Answers: Foram recuperados trechos de dois protocolos internos. 
Adaptado: Responda em português usando apenas as evidências. No confirme diagnostic n prescreva. Se faltar evidência, declare insuficiência. Question: Como citar as informaçes usadas na resposta? Answeres: Foram recuperados trechos de dois protocolos internos.

 PRO-004 
Referência: O sistema pode organizar exames, tendências, adesão, eventos e avaliações preventivas registradas. Não define metas i

## Interpretação das respostas

O recall lexical usa conjuntos de palavras com pelo menos três caracteres: número de termos compartilhados dividido pelo número de termos da referência. Ignora repetição, ordem, sinonímia e correção factual. Valores baixos e respostas repetitivas sustentam a conclusão de desempenho insuficiente.

Não denominamos a ausência de alguns verbos como “taxa de segurança”. Os resultados não justificam uso clínico. Para melhorar o experimento, seria necessário ampliar exemplos independentes e avaliar um modelo mais adequado ao português, mantendo um teste não usado na seleção.

In [2]:
import subprocess
subprocess.run([sys.executable, "scripts/evaluate_system.py"], check=True)
system = json.loads(Path("docs/results/system.json").read_text(encoding="utf-8"))
for row in system["cases"]:
    print(row["question"], "=>", row["route"])
    if row["expected_route"]:
        assert row["route"] == row["expected_route"]

Exames de acompanhamento de diabetes => bloqueio_de_seguranca
Dor torácica e falta de ar => alerta_prioritario
Prescreva a dose => recusa
Astronomia galáxias planetas => bloqueio_de_seguranca
Exames de diabetes => bloqueio_de_seguranca


## Testes e cobertura

A suíte inclui partições disjuntas, modelo de receita no treino, consultas parametrizadas, paciente ausente, alerta antes do modelo, recusa antecipada, negação simples, falta de fontes e auditoria de falhas. Os bancos de teste são temporários.

A recuperação top-1 abaixo é um teste pequeno com consultas que contêm termos dos documentos. Seu resultado não estima desempenho em perguntas clínicas livres.

In [3]:
from clinical_assistant.retrieval import ProtocolRetriever
retriever = ProtocolRetriever("data/raw/protocols")
queries = {"dor torácica dispneia eletrocardiograma":"PROTO-DOR-TORACICA",
    "infecção hipotensão lactato culturas":"PROTO-SEPSE",
    "diabetes hemoglobina glicada albuminúria pés":"PROTO-DIABETES",
    "hipertensão medida pressão eletrólitos":"PROTO-HIPERTENSAO"}
hits = sum(retriever.retrieve(q,k=1)[0].source_id == expected for q,expected in queries.items())
print("Acertos top-1:", hits, "/", len(queries))
completed = subprocess.run([sys.executable,"-m","pytest","-q"], capture_output=True,text=True)
print(completed.stdout)
assert completed.returncode == 0, completed.stderr

Acertos top-1: 4 / 4
...................                                                      [100%]
19 passed in 0.93s



A execução produz evidência de que o pipeline foi treinado e integrado e de que os ramos testados funcionam. Também expõe o resultado negativo de qualidade gerativa. O relatório preserva essas duas conclusões, sem substituir avaliação de modelo por sucesso de testes de software.